# 1. Install LLaMA-Factory and Dependencies
We need to install the `llamatuner` (LLaMA-Factory) package along with `bitsandbytes` for quantization and `peft` for LoRA support.

In [ ]:
# 1. Install LLaMA-Factory correctly
!rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install .[metrics,bitsandbytes,peft] --quiet
!pip install accelerate transformers modelscope hf_transfer --quiet
# Switch back to content directory
%cd ..

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 654, done.
remote: Counting objects: 100% (654/654), done.
remote: Compressing objects: 100% (492/492), done.
remote: Total 654 (delta 156), reused 422 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (654/654), 5.29 MiB | 9.73 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/content/LLaMA-Factory
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 620.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 11.6 MB/s eta 0:00:00
   ━━━━

In [ ]:
pip install transformers==5.2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.4.0
    Uninstalling transformers-5.4.0:
      Successfully uninstalled transformers-5.4.0




```
# This is formatted as code
```

### 1.5 Prepare Custom Dataset
To use your own data, create a JSON file and register it in `data/dataset_info.json`. Below is an example of how to format your data and register it.

In [ ]:
import json
import os

# 1. Define the target data directory
data_dir = "/content/LLaMA-Factory/data"
os.makedirs(data_dir, exist_ok=True)

# 2. Create the sample dataset
sample_data = [
  {
    "instruction": "<QueryRefiner>\nuser\n<title>\nراهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده\n</title>\n\n<chat_history>\n\n</chat_history>\n\n<user_input>\nسلام ب نظرت میتونی کمکی بکنی؟\nدرخواست کارت دادم بیشتراز ده روزه هنوز ب دستم نرسیده\n</user_input>\n\nOutput language:PERSIAN\n\nRefine the <user_input> based on the Instructions and Examples. Let's think step by step to generate the refined output.\nassistant\n",
    "input": "",
    "output": "{\"output\": \"سلام. به نظرت میتونی کمکی بکنی؟ من درخواست صدور کارت بانکت دادم و بیش از ده روزه که هنوز به دستم نرسیده.\"}"
  },
  {
    "instruction": "<QueryRefiner>\nuser\n<title>\nراهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده\n</title>\n\n<chat_history>\n<chat_history>\n\nUser: سلام\nChatbot: سلام! 😊 خوش آمدید به اپلیکیشن بانکت. چطور می‌توانم به شما کمک کنم؟\n</chat_history>\n\n</chat_history>\n\n<user_input>\nنمیشه حساب کاربری ساختش\n</user_input>\n\nOutput language:PERSIAN\n\nRefine the <user_input> based on the Instructions and Examples. Let's think step by step to generate the refined output.\nassistant\n",
    "input": "",
    "output": "{\"output\": \"نمیشه حساب کاربری در اپلیکیشن بانکت ساخت؟\"}"
  },
  {
    "instruction": "<QueryRefiner>\nuser\n<title>\nراهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده\n</title>\n\n<chat_history>\n<form_information>\n\nChatbot: در چه موردی سوال دارید؟\nUser: ارتباط با اپراتور\n</form_information>\n\n<chat_history>\n\nUser: خسته نباش،اگه میخواستم برم پست میرفتم بانک\nChatbot: متوجه هستم که ممکن است از روند دریافت کارت بانکت از طریق پست ناراضی باشید. در حال حاضر، ارسال کارت بانکت تنها از طریق پست انجام می‌شود و امکان مراجعه به شعبه برای دریافت کارت وجود ندارد. همچنین، کد رهگیری پستی همزمان با ارسال کارت به شما ارسال می‌شود. اگر کد رهگیری را دریافت نکرده‌اید، باید اطمینان حاصل کنید که حداقل 14 روز کاری از تاریخ افتتاح حساب شما گذشته باشد و سپس با کارت ملی به اداره پست مراجعه کنید. متاسفانه، واحد پشتیبانی بانکت به کد رهگیری پستی دسترسی ندارد و نمی‌تواند در این زمینه کمکی کند.\n</chat_history>\n\n</chat_history>\n\n<user_input>\nن ارسال نشده\n</user_input>\n\nOutput language:PERSIAN\n\nRefine the <user_input> based on the Instructions and Examples. Let's think step by step to generate the refined output.\nassistant\n",
    "input": "",
    "output": "{\"output\": \"کارت بانکت من ارسال نشده.\"}"
  }
]

# Multiply data for better demonstration of training cycle (optional)
sample_data = sample_data * 5

custom_data_path = os.path.join(data_dir, "custom_data.json")
with open(custom_data_path, "w", encoding="utf-8") as f:
    json.dump(sample_data, f, indent=2)

# 3. Define the registration info
dataset_info = {
    "my_custom_data": {
        "file_name": "custom_data.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output"
        }
    }
}

dataset_info_path = os.path.join(data_dir, "dataset_info.json")
with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)

print(f"Dataset and registration info saved to {data_dir}.")
print("You can now proceed to the fine-tuning cell.")

Dataset and registration info saved to /content/LLaMA-Factory/data.
You can now proceed to the fine-tuning cell.


# 2. Configure and Run Fine-tuning
We will define the training parameters for Qwen using the LoRA method. This example uses a placeholder dataset `identity`. You can replace `dataset: identity` with your own dataset name registered in the data folder.

In [ ]:
import os
import sys
import json

# 1. Setup paths
llm_path = "/content/LLaMA-Factory"
src_path = os.path.join(llm_path, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# 2. Define training configuration
train_args = {
  "stage": "sft",
  "do_train": True,
  "model_name_or_path": "Qwen/Qwen3.5-2B",  #  "Qwen/Qwen2.5-0.5B",
  "dataset": "my_custom_data",
  "dataset_dir": "/content/LLaMA-Factory/data",
  "template": "qwen",  # "qwen",
  "finetuning_type": "lora",
  "lora_target": "all",
  "output_dir": "qwen_lora_checkpoint",
  "overwrite_output_dir": True,
  "per_device_train_batch_size": 2,
  "gradient_accumulation_steps": 4,
  "lr_scheduler_type": "cosine",
  "logging_steps": 1000,
  "save_steps": 1000,
  "learning_rate": 1e-3,
  "num_train_epochs": 2.0,
  "plot_loss": True,
  "fp16": True
}

try:
    # 3. Import and run using the API
    from llamafactory.train.tuner import run_exp
    print("Successfully imported llamafactory engine.")
    print("Starting fine-tuning...")
    run_exp(train_args)
    print("\nTraining completed successfully!")
except Exception as e:
    print(f"An error occurred: {e}")
    import traceback
    traceback.print_exc()

[INFO|training_args.py:1745] 2026-04-25 08:57:49,458 >> PyTorch: setting up devices


Successfully imported llamafactory engine.
Starting fine-tuning...
[INFO|2026-04-25 08:57:49] llamafactory.hparams.parser:505 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16


[INFO|configuration_utils.py:670] 2026-04-25 08:57:49,784 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json
[INFO|configuration_utils.py:742] 2026-04-25 08:57:49,787 >> Model config Qwen3_5Config {
  "architectures": [
    "Qwen3_5ForConditionalGeneration"
  ],
  "image_token_id": 248056,
  "model_type": "qwen3_5",
  "text_config": {
    "attention_bias": false,
    "attention_dropout": 0.0,
    "attn_output_gate": true,
    "bos_token_id": null,
    "dtype": "bfloat16",
    "eos_token_id": 248044,
    "full_attention_interval": 4,
    "head_dim": 256,
    "hidden_act": "silu",
    "hidden_size": 2048,
    "initializer_range": 0.02,
    "intermediate_size": 6144,
    "layer_types": [
      "linear_attention",
      "linear_attention",
      "linear_attention",
      "full_attention",
      "linear_attention",
      "linear_attention",
      "linear_attention",
    

[INFO|2026-04-25 08:58:03] llamafactory.data.loader:144 >> Loading dataset custom_data.json...
training example:
input_ids:
[248045, 8678, 198, 2523, 513, 1167, 16451, 11, 3354, 539, 52540, 14388, 13, 1394, 513, 264, 10631, 17313, 13, 248046, 198, 248045, 846, 198, 27, 2766, 3812, 10140, 29, 198, 846, 198, 26451, 29, 198, 148310, 153439, 196456, 232955, 163123, 149083, 31246, 206666, 160921, 155417, 25, 173796, 162937, 79415, 160921, 151242, 36281, 218294, 160177, 86288, 152136, 208167, 156192, 180292, 244551, 196456, 232955, 163123, 149083, 31246, 206666, 160921, 155417, 25, 173796, 162937, 79415, 160921, 151242, 36281, 218294, 160177, 86288, 152136, 208167, 156192, 180292, 198, 510, 2034, 29, 271, 27, 9398, 19197, 29, 271, 510, 9398, 19197, 29, 271, 27, 846, 5715, 29, 198, 151513, 26942, 152191, 149227, 152741, 180209, 153550, 151242, 26942, 150136, 13978, 152033, 198, 148835, 168104, 238567, 42858, 157191, 159843, 149527, 175356, 160284, 51909, 163323, 215761, 26942, 151679, 9873, 4

[INFO|configuration_utils.py:670] 2026-04-25 08:58:04,277 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json
[INFO|configuration_utils.py:742] 2026-04-25 08:58:04,279 >> Model config Qwen3_5Config {
  "architectures": [
    "Qwen3_5ForConditionalGeneration"
  ],
  "image_token_id": 248056,
  "model_type": "qwen3_5",
  "text_config": {
    "attention_bias": false,
    "attention_dropout": 0.0,
    "attn_output_gate": true,
    "bos_token_id": null,
    "dtype": "bfloat16",
    "eos_token_id": 248044,
    "full_attention_interval": 4,
    "head_dim": 256,
    "hidden_act": "silu",
    "hidden_size": 2048,
    "initializer_range": 0.02,
    "intermediate_size": 6144,
    "layer_types": [
      "linear_attention",
      "linear_attention",
      "linear_attention",
      "full_attention",
      "linear_attention",
      "linear_attention",
      "linear_attention",
    

...

Step,Training Loss


[INFO|trainer.py:3797] 2026-04-25 09:02:10,899 >> Saving model checkpoint to qwen_lora_checkpoint/checkpoint-26
[INFO|tokenization_utils_base.py:3224] 2026-04-25 09:02:11,313 >> chat template saved in qwen_lora_checkpoint/checkpoint-26/chat_template.jinja
[INFO|tokenization_utils_base.py:2078] 2026-04-25 09:02:11,314 >> tokenizer config file saved in qwen_lora_checkpoint/checkpoint-26/tokenizer_config.json
[INFO|tokenization_utils_base.py:3224] 2026-04-25 09:02:11,879 >> chat template saved in qwen_lora_checkpoint/checkpoint-26/chat_template.jinja
[INFO|tokenization_utils_base.py:2078] 2026-04-25 09:02:11,881 >> tokenizer config file saved in qwen_lora_checkpoint/checkpoint-26/tokenizer_config.json
[INFO|processing_utils.py:852] 2026-04-25 09:02:12,322 >> chat template saved in qwen_lora_checkpoint/checkpoint-26/chat_template.jinja
[INFO|processing_utils.py:870] 2026-04-25 09:02:13,586 >> processor saved in qwen_lora_checkpoint/checkpoint-26/processor_config.json
[INFO|trainer.py:1863]

***** train metrics *****
  epoch                    =        2.0
  total_flos               =   897824GF
  train_loss               =     0.8045
  train_runtime            = 0:03:54.38
  train_samples_per_second =      0.853
  train_steps_per_second   =      0.111
[WARNING|2026-04-25 09:02:18] llamafactory.extras.ploting:149 >> No metric loss to plot.
[WARNING|2026-04-25 09:02:18] llamafactory.extras.ploting:149 >> No metric eval_loss to plot.
[WARNING|2026-04-25 09:02:18] llamafactory.extras.ploting:149 >> No metric eval_accuracy to plot.


[INFO|modelcard.py:266] 2026-04-25 09:02:18,231 >> Dropping the following result as it does not have all the necessary fields:
{'task': {'name': 'Causal Language Modeling', 'type': 'text-generation'}}



Training completed successfully!


### 3. Load and merge the Lora model and Run Inference
Now we load the lora model and  merge it with base model and process the specific Persian input provided.

In [ ]:

# The specific prompt requested
prompt = "<QueryRefiner>\nuser\n<title>\nراهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده\n</title>\n\n<chat_history>\n<form_information>\n\nChatbot: در چه موردی سوال دارید؟\nUser: ارتباط با اپراتور\n</form_information>\n\n<chat_history>\n\nUser: خسته نباش،اگه میخواستم برم پست میرفتم بانک\nChatbot: متوجه هستم که ممکن است از روند دریافت کارت بانکت از طریق پست ناراضی باشید. در حال حاضر، ارسال کارت بانکت تنها از طریق پست انجام می‌شود و امکان مراجعه به شعبه برای دریافت کارت وجود ندارد. همچنین، کد رهگیری پستی همزمان با ارسال کارت به شما ارسال می‌شود. اگر کد رهگیری را دریافت نکرده‌اید، باید اطمینان حاصل کنید که حداقل 14 روز کاری از تاریخ افتتاح حساب شما گذشته باشد و سپس با کارت ملی به اداره پست مراجعه کنید. متاسفانه، واحد پشتیبانی بانکت به کد رهگیری پستی دسترسی ندارد و نمی‌تواند در این زمینه کمکی کند.\n</chat_history>\n\n</chat_history>\n\n<user_input>\nن ارسال نشده\n</user_input>\n\nOutput language:PERSIAN\n\nRefine the <user_input> based on the Instructions and Examples. Let's think step by step to generate the refined output.\nassistant\n"




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: qwen_merged_model is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
import os
import sys

# Ensure LLaMA-Factory is in path for exporting
src_path = "/content/LLaMA-Factory/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from llamafactory.train.tuner import export_model

# Configuration for merging the LoRA adapter
export_args = {
    "model_name_or_path": "Qwen/Qwen2.5-0.5B",
    "adapter_name_or_path": "/content/qwen_lora_checkpoint",
    "template": "qwen",
    "finetuning_type": "lora",
    "export_dir": "qwen_merged_model",
    "export_size": 2,
    "export_device": "cpu",
    "export_legacy_format": False
}

print("Merging model... this may take a minute.")
try:
    export_model(export_args)
    print("Success: Merged model saved to 'qwen_merged_model'")
except Exception as e:
    print(f"Error during merging: {e}")

[INFO|configuration_utils.py:667] 2026-04-22 07:03:04,964 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/config.json
[INFO|configuration_utils.py:739] 2026-04-22 07:03:04,965 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "ful

Merging model... this may take a minute.


[INFO|configuration_utils.py:667] 2026-04-22 07:03:06,659 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/config.json
[INFO|configuration_utils.py:739] 2026-04-22 07:03:06,661 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "ful

[INFO|2026-04-22 07:03:07] llamafactory.data.template:144 >> Replace eos token: <|im_end|>.


[INFO|configuration_utils.py:667] 2026-04-22 07:03:08,014 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/config.json
[INFO|configuration_utils.py:739] 2026-04-22 07:03:08,015 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "ful

...

[INFO|2026-04-22 07:03:08] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-04-22 07:03:11] llamafactory.model.adapter:144 >> Merged 1 adapter(s).
[INFO|2026-04-22 07:03:11] llamafactory.model.adapter:144 >> Loaded adapter(s): /content/qwen_lora_checkpoint
[INFO|2026-04-22 07:03:11] llamafactory.model.loader:144 >> all params: 494,032,768
[INFO|2026-04-22 07:03:11] llamafactory.train.tuner:144 >> Convert model dtype to: torch.bfloat16.


[INFO|configuration_utils.py:432] 2026-04-22 07:03:11,684 >> Configuration saved in qwen_merged_model/config.json
[INFO|configuration_utils.py:803] 2026-04-22 07:03:11,686 >> Configuration saved in qwen_merged_model/generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[INFO|modeling_utils.py:3438] 2026-04-22 07:03:34,754 >> Model weights saved in qwen_merged_model/model.safetensors
[INFO|tokenization_utils_base.py:3327] 2026-04-22 07:03:34,757 >> chat template saved in qwen_merged_model/chat_template.jinja
[INFO|tokenization_utils_base.py:2181] 2026-04-22 07:03:34,757 >> tokenizer config file saved in qwen_merged_model/tokenizer_config.json


[INFO|2026-04-22 07:03:34] llamafactory.train.tuner:144 >> Ollama modelfile saved in qwen_merged_model/Modelfile
Success: Merged model saved to 'qwen_merged_model'


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "/content/qwen_merged_model"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading merged model onto {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)
# prompt = "<QueryRefiner>\nuser\n<title>\nراهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده\n</title>\n\n<chat_history>\n<form_information>\n\nChatbot: در چه موردی سوال دارید؟\nUser: ارتباط با اپراتور\n</form_information>\n\n<chat_history>\n\nUser: خسته نباش،اگه میخواستم برم پست میرفتم بانک\nChatbot: متوجه هستم که ممکن است از روند دریافت کارت بانکت از طریق پست ناراضی باشید. در حال حاضر، ارسال کارت بانکت تنها از طریق پست انجام می‌شود و امکان مراجعه به شعبه برای دریافت کارت وجود ندارد. همچنین، کد رهگیری پستی همزمان با ارسال کارت به شما ارسال می‌شود. اگر کد رهگیری را دریافت نکرده‌اید، باید اطمینان حاصل کنید که حداقل 14 روز کاری از تاریخ افتتاح حساب شما گذشته باشد و سپس با کارت ملی به اداره پست مراجعه کنید. متاسفانه، واحد پشتیبانی بانکت به کد رهگیری پستی دسترسی ندارد و نمی‌تواند در این زمینه کمکی کند.\n</chat_history>\n\n</chat_history>\n\n<user_input>\nن ارسال نشده\n</user_input>\n\nOutput language:PERSIAN\n\nRefine the <user_input> based on the Instructions and Examples. Let's think step by step to generate the refined output.\nassistant\n"
prompt = "<QueryRefiner>\nuser\n<title>\nراهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده\n</title>\n\n<chat_history>\n<form_information>\n\nChatbot: در چه موردی سوال دارید؟\nUser: ارتباط با اپراتور\n</form_information>\n\n<chat_history>\n\nUser: خسته نباش،اگه میخواستم برم پست میرفتم بانک\nChatbot: متوجه هستم که ممکن است از روند دریافت کارت بانکت از طریق پست ناراضی باشید. در حال حاضر، ارسال کارت بانکت تنها از طریق پست انجام می‌شود و امکان مراجعه به شعبه برای دریافت کارت وجود ندارد. همچنین، کد رهگیری پستی همزمان با ارسال کارت به شما ارسال می‌شود. اگر کد رهگیری را دریافت نکرده‌اید، باید اطمینان حاصل کنید که حداقل 14 روز کاری از تاریخ افتتاح حساب شما گذشته باشد و سپس با کارت ملی به اداره پست مراجعه کنید. متاسفانه، واحد پشتیبانی بانکت به کد رهگیری پستی دسترسی ندارد و نمی‌تواند در این زمینه کمکی کند.\n</chat_history>\n\n</chat_history>\n\n<user_input>\اپلیکیشن رو از کجا دانلود کنم؟ \n</user_input>\n\nOutput language:PERSIAN\n\nRefine the <user_input> based on the Instructions and Examples. Let's think step by step to generate the refined output.\nassistant\n"
# Prepare input
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Generate
print("Generating refined output...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        repetition_penalty=1.1
    )

# Decode and show result
full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract only the assistant part
refined_output = full_output.split("assistant")[-1].strip()

print("\n--- FINAL REFINED OUTPUT ---")
print(refined_output)

[INFO|configuration_utils.py:665] 2026-04-22 07:05:35,678 >> loading configuration file /content/qwen_merged_model/config.json
[INFO|configuration_utils.py:739] 2026-04-22 07:05:35,681 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    

Loading merged model onto cuda...


[INFO|configuration_utils.py:665] 2026-04-22 07:05:37,858 >> loading configuration file /content/qwen_merged_model/config.json
[INFO|configuration_utils.py:739] 2026-04-22 07:05:37,860 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[INFO|configuration_utils.py:965] 2026-04-22 07:05:38,744 >> loading configuration file /content/qwen_merged_model/generation_config.json
[INFO|configuration_utils.py:1014] 2026-04-22 07:05:38,746 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": false,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}

[WARNING|utils.py:2109] 2026-04-22 07:05:38,755 >> Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generating refined output...

--- FINAL REFINED OUTPUT ---
{"output": "پلیکیشن من در اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده راهنمای جامع اپلیکیشن بانکت: خدمات نوین بانکی و سرویس‌های ارزش افزوده."}


### 4. Export results to Google Drive
We will mount Google Drive and copy the model checkpoints and the LLaMA-Factory setup to ensure your work is saved.

In [ ]:
from google.colab import drive
# Adding force_remount=True can sometimes help resolve the 'mount failed' error
drive.mount('/content/drive', force_remount=True)

ValueError: mount failed

In [ ]:
import os

# Check if drive is mounted before proceeding
if os.path.exists('/content/drive/MyDrive'):
    # Define the backup destination
    backup_path = "/content/drive/MyDrive/LLaMA_Factory_Backup"
    os.makedirs(backup_path, exist_ok=True)

    print(f"Backing up files to: {backup_path}")

    # Copy the LoRA checkpoints
    !cp -r /content/qwen_lora_checkpoint "{backup_path}/"

    # Copy the merged model
    !cp -r /content/qwen_merged_model "{backup_path}/"

    # Copy the LLaMA-Factory directory
    !cp -r /content/LLaMA-Factory "{backup_path}/"

    print("Backup complete!")
else:
    print("Error: Google Drive is not mounted. Please run the previous cell and authorize access.")

Backing up files to: /content/drive/MyDrive/LLaMA_Factory_Backup
Backup complete!


In [ ]:
# Zip the LoRA checkpoint and LLaMA-Factory folders into one archive
!zip -r fine_tuning_results.zip /content/qwen_lora_checkpoint /content/LLaMA-Factory

# Show the size of the resulting zip file
!du -sh fine_tuning_results.zip

  adding: content/qwen_lora_checkpoint/ (stored 0%)
  adding: content/qwen_lora_checkpoint/trainer_state.json (deflated 58%)
  adding: content/qwen_lora_checkpoint/training_args.bin (deflated 53%)
  adding: content/qwen_lora_checkpoint/trainer_log.jsonl (deflated 28%)
  adding: content/qwen_lora_checkpoint/README.md (deflated 46%)
  adding: content/qwen_lora_checkpoint/all_results.json (deflated 38%)
  adding: content/qwen_lora_checkpoint/tokenizer.json (deflated 81%)
  adding: content/qwen_lora_checkpoint/chat_template.jinja (deflated 71%)
  adding: content/qwen_lora_checkpoint/tokenizer_config.json (deflated 59%)
  adding: content/qwen_lora_checkpoint/adapter_model.safetensors (deflated 8%)
  adding: content/qwen_lora_checkpoint/adapter_config.json (deflated 59%)
  adding: content/qwen_lora_checkpoint/train_results.json (deflated 38%)
  adding: content/qwen_lora_checkpoint/checkpoint-4/ (stored 0%)
  adding: content/qwen_lora_checkpoint/checkpoint-4/trainer_state.json (deflated 56%)
